# OptiCrop: Crop Recommendation Model Exploration & Comparison

This notebook covers the exploratory data analysis, preprocessing, and evaluation of multiple machine learning models to recommend suitable crops based on soil and climatic features.

## 1. Import Dependencies and Load Data

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

%matplotlib inline
sns.set_theme(style="whitegrid")

In [ ]:
# Load dataset
df = pd.read_csv("../data/Crop_recommendation.csv")
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Descriptive statistics
df.describe()

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Distribution of crop label counts
plt.figure(figsize=(12, 6))
sns.countplot(y="label", data=df, order=df["label"].value_counts().index, palette="viridis")
plt.title("Distribution of Crops in the Dataset")
plt.xlabel("Count")
plt.ylabel("Crop")
plt.show()

In [ ]:
# Correlation heatmap of numerical features
plt.figure(figsize=(10, 8))
numerical_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.show()

## 3. Data Split and Preprocessing

In [ ]:
X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 4. Model Comparisons

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

model_names = []
accuracies = []

for name, model in models.items():
    # Pipeline with StandardScaler
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    model_names.append(name)
    accuracies.append(acc)
    print(f"{name} Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, digits=4))
    print("-" * 60)

In [ ]:
# Visualizing model comparison
plt.figure(figsize=(10, 5))
sns.barplot(x=model_names, y=accuracies, palette="mako")
plt.title("Machine Learning Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0.9, 1.0)
for idx, val in enumerate(accuracies):
    plt.text(idx, val + 0.005, f"{val*100:.2f}%", ha='center', fontweight='bold')
plt.show()

## 5. Model Serialization (Persistence)

In [ ]:
import pickle

# Retrain the best performing model (Random Forest in pipeline) on full dataset
best_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

best_pipeline.fit(X, y)

# Save model
os.makedirs("../models", exist_ok=True)
with open("../models/crop_recommendation_model.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)

print("Trained model pipeline successfully serialized to models/crop_recommendation_model.pkl")